In [1]:
import math

def extended_gcd(a, b):
    """
    扩展欧几里得算法，用于求解线性同余方程。
    Расширенный алгоритм Евклида для решения линейных сравнений.
    """
    if a == 0:
        return b, 0, 1
    d, x1, y1 = extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return d, x, y

def solve_linear_congruence(A, B, M):
    """
    求解 Ax ≡ B (mod M) 类型的方程。
    Решение сравнения вида Ax ≡ B (mod M).
    """
    d, x0, y0 = extended_gcd(A, M)
    if B % d != 0:
        return None  # Решений нет (无解)
    x0 = (x0 * (B // d)) % (M // d)
    return x0

def pollard_rho_discrete_log(p, a, r, b):
    """
    离散对数波拉德 rho 方法。
    р-Метод Полларда для задачи дискретного логарифмирования.
    """
    # 按照实验定义的分段映射函数 f(c)
    # Определение ветвящейся функции f(c) согласно лабораторной работе
    def f(c, u, v):
        # 实验报告示例中使用 p/2 作为分界点
        # В примере из отчета p/2 используется как точка раздела
        if c < p // 2:
            # f(c) = ac, log_a(f(c)) = log_a(c) + 1
            return (a * c) % p, (u + 1) % r, v % r
        else:
            # f(c) = bc, log_a(f(c)) = log_a(c) + x
            return (b * c) % p, u % r, (v + 1) % r

    # 1. 选择随机初值 u, v 并计算 c
    # 1. Выбрать произвольные целые числа u, v и положить c = a^u * b^v (mod p)
    u_c, v_c = 2, 2  # 采用实验报告中的初值
    c = (pow(a, u_c, p) * pow(b, v_c, p)) % p

    u_d, v_d = u_c, v_c
    d = c

    print(f"{'Step':<6} {'c':<10} {'log_a(c)':<15} {'d':<10} {'log_a(d)':<15}")
    print("-" * 60)

    # 2. 执行步进直到 c ≡ d (mod p)
    # 2. Выполнять шаги до получения равенства c ≡ d (mod p)
    # 使用 Floyd 判圈算法 (龟兔赛跑)
    for i in range(1, 1000): # 设置最大步数防止死循环
        # c 每次走一步
        c, u_c, v_c = f(c, u_c, v_c)
        # d 每次走两步
        d, u_d, v_d = f(d, u_d, v_d)
        d, u_d, v_d = f(d, u_d, v_d)

        print(f"{i:<6} {c:<10} {u_c}+{v_c}x{' ':<6} {d:<10} {u_d}+{v_d}x")

        if c == d:
            # 3. 建立线性方程: u_c + v_c*x ≡ u_d + v_d*x (mod r)
            # 3. Приравняв логарифмы, вычислить x решением сравнения (v_d - v_c)x ≡ (u_c - u_d) (mod r)
            A = (v_d - v_c) % r
            B = (u_c - u_d) % r

            x = solve_linear_congruence(A, B, r)
            return x

    return "Решений нет"



In [2]:
# 样例 (Пример):
# 使用实验报告中的数据: 10^x ≡ 64 (mod 107), r = 53
# Используем данные из примера: 10^x ≡ 64 (mod 107), r = 53
p_val = 107
a_val = 10
r_val = 53
b_val = 64

result = pollard_rho_discrete_log(p_val, a_val, r_val, b_val)

print("-" * 60)
print(f"结论 (Результат): x = {result}")
if result is not None:
    print(f"验证 (Проверка): {a_val}^{result} ≡ {pow(a_val, result, p_val)} (mod {p_val})")

Step   c          log_a(c)        d          log_a(d)       
------------------------------------------------------------
1      40         3+2x       79         4+2x
2      79         4+2x       56         5+3x
3      27         4+3x       75         5+5x
4      56         5+3x       3          5+7x
5      53         5+4x       86         7+7x
6      75         5+5x       42         8+8x
7      92         5+6x       23         9+9x
8      3          5+7x       53         11+9x
9      30         6+7x       92         11+11x
10     86         7+7x       30         12+12x
11     47         7+8x       47         13+13x
------------------------------------------------------------
结论 (Результат): x = 20
验证 (Проверка): 10^20 ≡ 64 (mod 107)
